In [1]:
# ============================================================================
# CELL 1: Environment Setup and Dependency Installation
# ============================================================================

import subprocess
import sys

def install_packages():
    """Install required Python packages for the object detection system."""
    packages = [
        'ultralytics',
        'opencv-python-headless',
        'gradio',
        'matplotlib',
        'seaborn',
        'pillow'
    ]
    
    for package in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"Installed: {package}")

install_packages()
print("Environment setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.3 MB/s eta 0:00:00
Installed: ultralytics
Installed: opencv-python-headless
Installed: gradio
Installed: matplotlib
Installed: seaborn
Installed: pillow
Environment setup complete.


In [2]:
# ============================================================================
# CELL 2: Import Required Libraries
# ============================================================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
import gradio as gr
from PIL import Image
import time
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib style for professional appearance
plt.style.use('seaborn-v0_8-darkgrid')

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch Version: 2.10.0+cu128
CUDA Available: True
Device: GPU


In [3]:
# ============================================================================
# CELL 3: Load Pre-trained YOLOv8 Model
# ============================================================================

def load_model(model_name='yolov8n.pt'):
    """
    Load the pre-trained YOLOv8 model.
    
    Parameters:
        model_name (str): Name of the model weights file
                         Options: yolov8n.pt (nano), yolov8s.pt (small),
                                 yolov8m.pt (medium), yolov8l.pt (large)
    
    Returns:
        YOLO: Loaded YOLO model instance
    """
    try:
        model = YOLO(model_name)
        print(f"Model loaded successfully: {model_name}")
        print(f"Number of classes: {len(model.names)}")
        print(f"Sample classes: {list(model.names.values())[:5]}...")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

# Load the model
model = load_model('yolov8n.pt')

# Store class information for reference
CLASS_NAMES = model.names if model else {}
NUM_CLASSES = len(CLASS_NAMES)

Model loaded successfully: yolov8n.pt
Number of classes: 80
Sample classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane']...


In [5]:
# ============================================================================
# CELL 4: Core Detection Functions
# ============================================================================

def detect_objects(image, confidence_threshold=0.5, iou_threshold=0.45):
    """
    Perform object detection on an input image using YOLO.
    
    Parameters:
        image (numpy.ndarray): Input image in RGB format
        confidence_threshold (float): Minimum confidence for detections
        iou_threshold (float): IoU threshold for NMS
    
    Returns:
        tuple: (annotated_image, detections_list, summary_text)
    """
    # Validate input
    if image is None:
        return None, [], "Error: No image provided"
    
    # Convert grayscale to RGB if needed
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    
    # Run inference
    results = model(image, conf=confidence_threshold, iou=iou_threshold)
    
    # Extract detections
    detections = []
    if len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        scores = results[0].boxes.conf.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        
        for box, score, cls in zip(boxes, scores, classes):
            x1, y1, x2, y2 = map(int, box)
            class_name = CLASS_NAMES[int(cls)]
            detections.append({
                'class': class_name,
                'class_id': int(cls),
                'confidence': float(score),
                'bbox': [x1, y1, x2, y2],
                'width': x2 - x1,
                'height': y2 - y1,
                'area': (x2 - x1) * (y2 - y1)
            })
    
    # Sort detections by confidence (highest first)
    detections.sort(key=lambda x: x['confidence'], reverse=True)
    
    # Create annotated image
    annotated_image = draw_detections(image.copy(), detections)
    
    # Generate summary
    summary = generate_summary(detections)
    
    return annotated_image, detections, summary

def draw_detections(image, detections):
    """
    Draw bounding boxes and labels on the image.
    
    Parameters:
        image (numpy.ndarray): Input image
        detections (list): List of detection dictionaries
    
    Returns:
        numpy.ndarray: Image with bounding boxes drawn
    """
    # Define colors for different classes
    colors = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        class_name = det['class']
        confidence = det['confidence']
        
        # Get color for this class
        color = colors[det['class_id'] % NUM_CLASSES]
        color_bgr = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
        
        # Draw bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), color_bgr, 2)
        
        # Prepare label text
        label = f"{class_name} {confidence:.2f}"
        
        # Calculate text size for background
        (text_width, text_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
        )
        
        # Draw label background
        cv2.rectangle(
            image,
            (x1, y1 - text_height - 10),
            (x1 + text_width, y1),
            color_bgr,
            -1
        )
        
        # Draw label text
        cv2.putText(
            image,
            label,
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )
    
    return image

def generate_summary(detections):
    """
    Generate a text summary of the detections.
    
    Parameters:
        detections (list): List of detection dictionaries
    
    Returns:
        str: Formatted summary text
    """
    if not detections:
        return "No objects detected in the image."
    
    # Count objects by class
    class_counts = {}
    for det in detections:
        class_name = det['class']
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
    
    # Build summary
    summary_parts = [
        f"Total objects detected: {len(detections)}",
        "=" * 40,
        "Detection Summary:"
    ]
    
    # Add class-wise counts
    for class_name, count in class_counts.items():
        summary_parts.append(f"  - {class_name}: {count}")
    
    summary_parts.append("=" * 40)
    summary_parts.append("Confidence Scores:")
    
    # Add confidence stats
    confidences = [det['confidence'] for det in detections]
    summary_parts.append(f"  - Highest: {max(confidences):.3f}")
    summary_parts.append(f"  - Average: {sum(confidences)/len(confidences):.3f}")
    summary_parts.append(f"  - Lowest: {min(confidences):.3f}")
    
    # Add bounding box statistics
    areas = [det['area'] for det in detections]
    summary_parts.append("=" * 40)
    summary_parts.append("Object Sizes:")
    summary_parts.append(f"  - Smallest: {min(areas):,} pixels")
    summary_parts.append(f"  - Largest: {max(areas):,} pixels")
    summary_parts.append(f"  - Average: {sum(areas)/len(areas):,.0f} pixels")
    
    return "\n".join(summary_parts)

In [6]:
# ============================================================================
# CELL 5: Performance Metrics Calculation
# ============================================================================

def calculate_performance_metrics():
    """
    Calculate and display performance metrics of the model.
    
    Returns:
        dict: Performance metrics
    """
    metrics = {}
    
    # Model size
    model_path = 'yolov8n.pt'
    if os.path.exists(model_path):
        size_bytes = os.path.getsize(model_path)
        metrics['model_size_mb'] = size_bytes / (1024 * 1024)
    
    # Model parameters
    metrics['total_classes'] = NUM_CLASSES
    metrics['device'] = 'GPU' if torch.cuda.is_available() else 'CPU'
    
    # Inference time test
    test_image = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    start_time = time.time()
    _ = model(test_image, conf=0.5)
    inference_time = time.time() - start_time
    metrics['inference_time_seconds'] = inference_time
    metrics['fps'] = 1 / inference_time
    
    # Display metrics
    print("\n" + "="*50)
    print("PERFORMANCE METRICS")
    print("="*50)
    print(f"Model Size: {metrics['model_size_mb']:.2f} MB")
    print(f"Total Classes: {metrics['total_classes']}")
    print(f"Device: {metrics['device']}")
    print(f"Inference Time: {metrics['inference_time_seconds']:.3f} seconds")
    print(f"FPS: {metrics['fps']:.1f}")
    print("="*50)
    
    return metrics

# Calculate and display metrics
performance_metrics = calculate_performance_metrics()


0: 640x640 (no detections), 7.7ms
Speed: 15.6ms preprocess, 7.7ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 640)

PERFORMANCE METRICS
Model Size: 6.25 MB
Total Classes: 80
Device: GPU
Inference Time: 1.670 seconds
FPS: 0.6


In [8]:
# ============================================================================
# CELL 6: Batch Processing Functions
# ============================================================================

def process_batch(image_list, confidence_threshold=0.5):
    """
    Process multiple images in batch mode.
    
    Parameters:
        image_list (list): List of images (numpy arrays)
        confidence_threshold (float): Confidence threshold for detections
    
    Returns:
        list: Results for each image
    """
    results = []
    
    for idx, image in enumerate(image_list):
        annotated, detections, summary = detect_objects(
            image, confidence_threshold
        )
        results.append({
            'index': idx,
            'detections': detections,
            'count': len(detections),
            'summary': summary
        })
    
    return results

def save_batch_results(results, output_dir='outputs'):
    """
    Save batch processing results.
    
    Parameters:
        results (list): Results from batch processing
        output_dir (str): Output directory
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Create summary file
    summary_data = []
    for result in results:
        summary_data.append({
            'image_index': result['index'],
            'object_count': result['count']
        })
    
    # Save as JSON
    json_path = os.path.join(output_dir, 'batch_summary.json')
    with open(json_path, 'w') as f:
        json.dump(summary_data, f, indent=2)
    
    print(f"Batch results saved to {output_dir}")

In [23]:
# ============================================================================
# CELL 7: Professional Gradio Web Interface (With Expandable Boxes)
# ============================================================================

def create_interface():
    """
    Create the professional Gradio web interface for the object detection system.
    Features a clean, modern design with comprehensive information display.
    
    Returns:
        gr.Interface: Configured Gradio interface
    """
    
    def process_image(image, confidence_threshold):
        """
        Wrapper function for Gradio interface with enhanced output formatting.
        
        Parameters:
            image (numpy.ndarray): Input image
            confidence_threshold (float): Confidence threshold
        
        Returns:
            tuple: (annotated_image, formatted_summary)
        """
        if image is None:
            return None, "ERROR: No image uploaded. Please upload an image to proceed."
        
        annotated, detections, summary = detect_objects(
            image, confidence_threshold
        )
        
        # Build comprehensive formatted summary
        formatted_summary = []
        
        # Header Section
        formatted_summary.append("=" * 70)
        formatted_summary.append("OBJECT DETECTION RESULTS")
        formatted_summary.append("=" * 70)
        formatted_summary.append("")
        
        # System Information Section
        formatted_summary.append("SYSTEM INFORMATION")
        formatted_summary.append("-" * 40)
        formatted_summary.append(f"Model Architecture   : YOLOv8 (You Only Look Once)")
        formatted_summary.append(f"Training Dataset     : COCO (Common Objects in Context)")
        formatted_summary.append(f"Total Classes        : {NUM_CLASSES}")
        formatted_summary.append(f"Confidence Threshold : {confidence_threshold:.2f}")
        formatted_summary.append(f"Inference Device     : {'GPU' if torch.cuda.is_available() else 'CPU'}")
        formatted_summary.append(f"Detection Timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        formatted_summary.append("")
        
        # Detection Summary Section
        if detections:
            formatted_summary.append("DETECTION SUMMARY")
            formatted_summary.append("-" * 40)
            formatted_summary.append(f"Total Objects Detected : {len(detections)}")
            formatted_summary.append("")
            
            # Class-wise breakdown
            class_counts = {}
            class_confidences = {}
            for det in detections:
                class_name = det['class']
                class_counts[class_name] = class_counts.get(class_name, 0) + 1
                if class_name not in class_confidences:
                    class_confidences[class_name] = []
                class_confidences[class_name].append(det['confidence'])
            
            formatted_summary.append("CLASS-WISE BREAKDOWN")
            formatted_summary.append("-" * 30)
            for class_name, count in class_counts.items():
                avg_conf = sum(class_confidences[class_name]) / len(class_confidences[class_name])
                formatted_summary.append(f"  {class_name:20s} : {count:3d} object(s)  (Avg Conf: {avg_conf:.3f})")
            formatted_summary.append("")
            
            # Confidence Statistics
            confidences = [det['confidence'] for det in detections]
            formatted_summary.append("CONFIDENCE STATISTICS")
            formatted_summary.append("-" * 30)
            formatted_summary.append(f"  Highest Confidence  : {max(confidences):.3f}")
            formatted_summary.append(f"  Average Confidence  : {sum(confidences)/len(confidences):.3f}")
            formatted_summary.append(f"  Lowest Confidence   : {min(confidences):.3f}")
            formatted_summary.append("")
            
            # Object Size Statistics
            areas = [det['area'] for det in detections]
            formatted_summary.append("OBJECT SIZE STATISTICS")
            formatted_summary.append("-" * 30)
            formatted_summary.append(f"  Smallest Object     : {min(areas):,} pixels")
            formatted_summary.append(f"  Largest Object      : {max(areas):,} pixels")
            formatted_summary.append(f"  Average Size        : {sum(areas)/len(areas):,.0f} pixels")
            formatted_summary.append("")
            
            # Detailed Detection List
            formatted_summary.append("DETAILED DETECTION LIST")
            formatted_summary.append("-" * 40)
            for i, det in enumerate(detections, 1):
                x1, y1, x2, y2 = det['bbox']
                width = x2 - x1
                height = y2 - y1
                formatted_summary.append(f"  {i:2d}. {det['class']:15s} | Confidence: {det['confidence']:.3f} | "
                                        f"Box: [{x1}, {y1}, {x2}, {y2}] | Size: {width}x{height}")
            
        else:
            formatted_summary.append("DETECTION STATUS")
            formatted_summary.append("-" * 40)
            formatted_summary.append("STATUS: No Objects Detected")
            formatted_summary.append("")
            formatted_summary.append("Possible Reasons:")
            formatted_summary.append("  1. The confidence threshold may be too high")
            formatted_summary.append("  2. The image may not contain recognizable objects")
            formatted_summary.append("  3. Objects may be too small or blurry")
            formatted_summary.append("")
            formatted_summary.append("Recommendations:")
            formatted_summary.append("  - Try lowering the confidence threshold")
            formatted_summary.append("  - Use images with clear, well-lit objects")
            formatted_summary.append("  - Ensure objects are from the 80 COCO classes")
        
        formatted_summary.append("")
        formatted_summary.append("=" * 70)
        formatted_summary.append("END OF REPORT")
        formatted_summary.append("=" * 70)
        
        return annotated, "\n".join(formatted_summary)
    
    # Professional CSS styling with responsive boxes
    custom_css = """
    /* Main container styling */
    .gradio-container {
        max-width: 1400px !important;
        margin: 0 auto !important;
        padding: 20px !important;
        background: #f8f9fa !important;
    }
    
    /* Interface box styling */
    .gradio-interface {
        background: white !important;
        border-radius: 16px !important;
        box-shadow: 0 8px 32px rgba(0,0,0,0.08) !important;
        padding: 24px !important;
    }
    
    /* Title styling */
    h1 {
        color: #1a237e !important;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
        font-size: 2.5rem !important;
        font-weight: 700 !important;
        text-align: center !important;
        margin-bottom: 8px !important;
        letter-spacing: 0.5px !important;
    }
    
    /* Input and output boxes - RESPONSIVE */
    .gr-box {
        border: 2px solid #e3e8ef !important;
        border-radius: 12px !important;
        padding: 16px !important;
        background: white !important;
        transition: border-color 0.3s ease !important;
        min-height: auto !important;
        height: auto !important;
    }
    
    .gr-box:hover {
        border-color: #1a237e !important;
    }
    
    /* Image display - ADJUSTABLE */
    .gr-image {
        border-radius: 8px !important;
        overflow: hidden !important;
        min-height: 400px !important;
        height: auto !important;
        max-height: 700px !important;
    }
    
    .gr-image img {
        object-fit: contain !important;
        width: 100% !important;
        height: auto !important;
        max-height: 700px !important;
    }
    
    /* Textbox/Report area - EXPANDABLE */
    .gr-textbox {
        min-height: 400px !important;
        height: auto !important;
        max-height: 800px !important;
    }
    
    .gr-textbox textarea {
        font-family: 'Consolas', 'Courier New', monospace !important;
        font-size: 0.9rem !important;
        line-height: 1.6 !important;
        background: #f8f9fa !important;
        border: 1px solid #e3e8ef !important;
        border-radius: 8px !important;
        padding: 16px !important;
        min-height: 400px !important;
        height: auto !important;
        max-height: 800px !important;
        color: #1a2332 !important;
        resize: vertical !important;
        overflow-y: auto !important;
    }
    
    /* Slider styling */
    .gr-slider input[type="range"] {
        accent-color: #1a237e !important;
        height: 6px !important;
    }
    
    .gr-slider label {
        color: #1a2332 !important;
        font-weight: 600 !important;
        font-size: 1rem !important;
    }
    
    /* Button styling */
    .gr-button {
        background: linear-gradient(135deg, #1a237e 0%, #283593 100%) !important;
        color: white !important;
        border: none !important;
        padding: 12px 32px !important;
        border-radius: 8px !important;
        font-weight: 600 !important;
        font-size: 1rem !important;
        transition: transform 0.2s ease, box-shadow 0.2s ease !important;
        cursor: pointer !important;
        width: 100% !important;
    }
    
    .gr-button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 6px 20px rgba(26, 35, 126, 0.3) !important;
    }
    
    .gr-button:active {
        transform: translateY(0px) !important;
    }
    
    /* Responsive columns */
    .gradio-row {
        display: flex !important;
        flex-wrap: wrap !important;
        gap: 20px !important;
    }
    
    .gradio-column {
        flex: 1 !important;
        min-width: 300px !important;
    }
    
    /* Footer styling */
    .footer {
        margin-top: 24px !important;
        padding-top: 16px !important;
        border-top: 1px solid #e3e8ef !important;
        text-align: center !important;
        color: #78909c !important;
        font-size: 0.9rem !important;
    }
    
    /* Responsive design */
    @media (max-width: 768px) {
        h1 {
            font-size: 1.8rem !important;
        }
        .gradio-container {
            padding: 10px !important;
        }
        .gradio-interface {
            padding: 16px !important;
        }
        .gr-image {
            min-height: 250px !important;
            max-height: 400px !important;
        }
        .gr-image img {
            max-height: 400px !important;
        }
        .gr-textbox textarea {
            min-height: 250px !important;
            max-height: 400px !important;
            font-size: 0.8rem !important;
        }
        .gradio-column {
            min-width: 100% !important;
        }
    }
    
    @media (min-width: 769px) and (max-width: 1024px) {
        .gr-image {
            min-height: 350px !important;
            max-height: 550px !important;
        }
        .gr-image img {
            max-height: 550px !important;
        }
        .gr-textbox textarea {
            min-height: 350px !important;
            max-height: 600px !important;
        }
    }
    
    /* Custom scrollbar */
    ::-webkit-scrollbar {
        width: 8px !important;
        height: 8px !important;
    }
    
    ::-webkit-scrollbar-track {
        background: #f1f1f1 !important;
        border-radius: 4px !important;
    }
    
    ::-webkit-scrollbar-thumb {
        background: #c1c7cd !important;
        border-radius: 4px !important;
    }
    
    ::-webkit-scrollbar-thumb:hover {
        background: #a0a8b0 !important;
    }
    
    /* Allow textarea resizing */
    .gr-textarea textarea {
        resize: vertical !important;
    }
    """
    
    # Create interface with enhanced layout
    interface = gr.Interface(
        fn=process_image,
        inputs=[
            gr.Image(
                label="UPLOAD IMAGE",
                type="numpy",
                sources=['upload', 'webcam'],
                interactive=True,
                height=400
            ),
            gr.Slider(
                minimum=0.1,
                maximum=0.9,
                value=0.5,
                step=0.05,
                label="CONFIDENCE THRESHOLD",
                info="Higher values = stricter detection, Lower values = more detections",
                interactive=True
            )
        ],
        outputs=[
            gr.Image(
                label="DETECTION RESULTS",
                type="numpy",
                height=500
            ),
            gr.Textbox(
                label="DETAILED REPORT",
                lines=25,
                max_lines=35,
                show_copy_button=True,
                interactive=False
            )
        ],
        title="CUSTOM OBJECT DETECTION SYSTEM",
        description="""
        <div style='text-align: center; margin-bottom: 20px;'>
            <span style='background: #1a237e; color: white; padding: 4px 16px; border-radius: 20px; font-weight: 600;'>
                YOLOv8
            </span>
            <span style='margin: 0 10px; color: #78909c;'>|</span>
            <span style='color: #546e7a;'>Pre-trained on COCO Dataset</span>
            <span style='margin: 0 10px; color: #78909c;'>|</span>
            <span style='color: #546e7a;'>80 Object Classes</span>
        </div>
        
        <div style='background: #f0f2f5; padding: 16px; border-radius: 8px; margin: 12px 0;'>
            <strong style='color: #1a2332;'>System Capabilities:</strong>
            <ul style='margin: 8px 0 0 0; padding-left: 20px; color: #37474f;'>
                <li>Detects 80 different object classes including people, vehicles, animals, and everyday items</li>
                <li>Real-time inference with high accuracy</li>
                <li>Adjustable confidence threshold for precision control</li>
                <li>Comprehensive detection reports with statistics</li>
            </ul>
        </div>
        
        <div style='background: #e8eaf6; padding: 12px; border-radius: 8px; margin: 12px 0;'>
            <strong style='color: #1a237e;'>Getting Started:</strong>
            <ol style='margin: 8px 0 0 0; padding-left: 20px; color: #37474f;'>
                <li>Upload an image using the upload area above</li>
                <li>Adjust the confidence threshold (0.5 recommended for balanced results)</li>
                <li>Click the Submit button to start detection</li>
                <li>View results on the right side with detailed summary</li>
            </ol>
        </div>
        
        <div style='background: #f3e5f5; padding: 12px; border-radius: 8px; margin: 12px 0;'>
            <strong style='color: #4a148c;'>Supported Object Categories:</strong>
            <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-top: 8px;'>
                <div>People</div>
                <div>Vehicles</div>
                <div>Animals</div>
                <div>Indoor Objects</div>
                <div>Food Items</div>
                <div>Sports Equipment</div>
                <div>Accessories</div>
                <div>Outdoor Objects</div>
            </div>
        </div>
        """,
        theme="default",
        css=custom_css,
        allow_flagging="never",
        live=False
    )
    
    return interface

In [24]:
# ============================================================================
# CELL 8: Main Execution and System Initialization
# ============================================================================

def create_directories():
    """Create necessary directories for the project."""
    directories = [
        'outputs',
        'outputs/images',
        'outputs/summaries',
        'test_images'
    ]
    for directory in directories:
        os.makedirs(directory, exist_ok=True)
    print("Directory structure created.")

def save_system_info():
    """Save system information to a JSON file."""
    info = {
        'system': {
            'model': 'YOLOv8n',
            'classes': NUM_CLASSES,
            'device': 'GPU' if torch.cuda.is_available() else 'CPU',
            'timestamp': datetime.now().isoformat()
        },
        'performance': performance_metrics
    }
    
    with open('outputs/system_info.json', 'w') as f:
        json.dump(info, f, indent=2)
    
    print("System information saved.")

def main():
    """Main execution function."""
    print("\n" + "="*60)
    print("CUSTOM OBJECT DETECTION SYSTEM")
    print("YOLOv8 - Pre-trained on COCO Dataset")
    print("="*60)
    
    # Create directories
    create_directories()
    
    # Save system information
    save_system_info()
    
    # Create interface
    print("\nInitializing web interface...")
    interface = create_interface()
    
    print("\n" + "="*60)
    print("SYSTEM READY")
    print("="*60)
    print(f"Total Classes: {NUM_CLASSES}")
    print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
    print("Output Directory: outputs/")
    print("\nLaunching interface...")
    print("="*60 + "\n")
    
    # Launch the interface
    interface.launch(
        share=True,
        debug=False,
        quiet=True
    )

# Run the main function
if __name__ == "__main__":
    main()


CUSTOM OBJECT DETECTION SYSTEM
YOLOv8 - Pre-trained on COCO Dataset
Directory structure created.
System information saved.

Initializing web interface...

SYSTEM READY
Total Classes: 80
Device: GPU
Output Directory: outputs/

Launching interface...

* Running on public URL: https://85df4c39994b527dd2.gradio.live


In [11]:
# ============================================================================
# CELL 9: Test Function (Optional - Run to verify system)
# ============================================================================

def run_test():
    """
    Run a test to verify the system is working correctly.
    Creates a simple test image and processes it.
    """
    print("\n" + "="*50)
    print("RUNNING SYSTEM TEST")
    print("="*50)
    
    # Create a test image
    test_image = np.ones((480, 640, 3), dtype=np.uint8) * 240
    
    # Draw some shapes (simulating objects)
    cv2.rectangle(test_image, (100, 150), (250, 350), (0, 0, 255), -1)
    cv2.circle(test_image, (150, 350), 30, (0, 0, 0), -1)
    cv2.circle(test_image, (200, 350), 30, (0, 0, 0), -1)
    cv2.rectangle(test_image, (400, 150), (550, 350), (255, 0, 0), -1)
    
    # Save test image
    test_path = 'test_images/test_image.jpg'
    cv2.imwrite(test_path, cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR))
    
    # Process test image
    annotated, detections, summary = detect_objects(test_image)
    
    # Save annotated image
    output_path = 'outputs/images/test_result.jpg'
    cv2.imwrite(output_path, cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
    
    # Save summary
    summary_path = 'outputs/summaries/test_summary.txt'
    with open(summary_path, 'w') as f:
        f.write(summary)
    
    print(f"Test completed successfully!")
    print(f"Test image saved: {test_path}")
    print(f"Result saved: {output_path}")
    print(f"Summary saved: {summary_path}")
    print(f"Objects detected: {len(detections)}")
    print("="*50)

# Uncomment to run test
# run_test()


0: 608x640 (no detections), 55.2ms
Speed: 12.7ms preprocess, 55.2ms inference, 0.7ms postprocess per image at shape (1, 3, 608, 640)

0: 608x640 1 person, 7.7ms
Speed: 4.0ms preprocess, 7.7ms inference, 32.8ms postprocess per image at shape (1, 3, 608, 640)

0: 608x640 1 clock, 8.1ms
Speed: 3.9ms preprocess, 8.1ms inference, 1.5ms postprocess per image at shape (1, 3, 608, 640)

0: 384x640 (no detections), 72.0ms
Speed: 3.9ms preprocess, 72.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 6.5ms
Speed: 2.8ms preprocess, 6.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 umbrella, 6.6ms
Speed: 2.1ms preprocess, 6.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 cat, 2 knifes, 7.0ms
Speed: 2.3ms preprocess, 7.0ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 cat, 6.2ms
Speed: 2.1ms preprocess, 6.2ms inference, 1.3ms postprocess